# CLUSTERING TASK - Wine

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
import seaborn as sns

# Load file
df = pd.read_csv(r'C:\Users\ASUS\Downloads\wine_clean.csv')
df

In [ ]:
df.info()

In [ ]:
# Ambil kolom numerik
X = df.select_dtypes(include=[np.number]).values
feature_names = df.select_dtypes(include=[np.number]).columns.tolist()

# Normalisasi data (penting untuk clustering)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"\nShape data: {X.shape}")
print(f"Fitur: {feature_names}")

In [ ]:
# Fungsi evaluasi clustering
def evaluate_clustering(labels, X, name):
    if len(set(labels)) < 2:
        return {"silhouette": -1, "davies_bouldin": None, "n_clusters": len(set(labels))}
    
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    return {"silhouette": sil, "davies_bouldin": db, "n_clusters": len(set(labels))}

results = []

In [ ]:
# Pisahkan fitur (semua kolom kecuali Class)
if 'Class' in df.columns:
    X = df.drop('Class', axis=1).select_dtypes(include=[np.number]).values
    y_true = df['Class'].values
    print(f"\nClass asli: {np.unique(y_true)}")
else:
    X = df.select_dtypes(include=[np.number]).values
    y_true = None

feature_names = df.drop('Class', axis=1).select_dtypes(include=[np.number]).columns.tolist() if 'Class' in df.columns else df.select_dtypes(include=[np.number]).columns.tolist()

print(f"Fitur: {feature_names}")
print(f"Shape X: {X.shape}")

# ========== FUNGSI EVALUASI ==========
def evaluate_clustering(labels, X, y_true=None):
    if len(set(labels)) < 2:
        return {"silhouette": -1, "davies_bouldin": None, "ari": -1, "n_clusters": len(set(labels))}
    
    sil = silhouette_score(X, labels)
    db = davies_bouldin_score(X, labels)
    
    ari = -1
    if y_true is not None and -1 not in labels:
        ari = adjusted_rand_score(y_true, labels)
    
    return {"silhouette": sil, "davies_bouldin": db, "ari": ari, "n_clusters": len(set(labels))}

results = []

### 1. Agglomerative Clustering

In [ ]:
Z = linkage(X, method='ward')

plt.figure(figsize=(12, 6))
dendrogram(Z)
plt.title('Dendrogram - Agglomerative Clustering')
plt.xlabel('Data Points')
plt.ylabel('Distance')
plt.show()

best_agglo = None
best_sil = -1
best_k = 0

for k in [2, 3, 4, 5]:
    labels = fcluster(Z, t=k, criterion='maxclust')
    ev = evaluate_clustering(labels, X, y_true)
    print(f"k={k}: Silhouette={ev['silhouette']:.3f}, DB={ev['davies_bouldin']:.3f}, ARI={ev['ari']:.3f}")
    if ev['silhouette'] > best_sil:
        best_sil = ev['silhouette']
        best_agglo = labels
        best_k = k

print(f"\nBest: k={best_k}, Silhouette={best_sil:.3f}")
results.append({"method": "Agglomerative", "k": best_k, "silhouette": best_sil, "labels": best_agglo})

### 2. K-Means Clustering

In [ ]:
inertias = []
sil_scores = []
K_range = range(2, 7)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X)
    inertias.append(kmeans.inertia_)
    sil_scores.append(silhouette_score(X, labels))

plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(K_range, inertias, 'bo-')
plt.xlabel('k')
plt.ylabel('Inertia')
plt.title('Elbow Method')

plt.subplot(1, 2, 2)
plt.plot(K_range, sil_scores, 'ro-')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.title('Silhouette Score')

plt.tight_layout()
plt.show()

best_k = K_range[np.argmax(sil_scores)]
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
kmeans_labels = kmeans.fit_predict(X)
best_sil = silhouette_score(X, kmeans_labels)

ev = evaluate_clustering(kmeans_labels, X, y_true)
print(f"Best K-Means: k={best_k}, Silhouette={ev['silhouette']:.3f}, DB={ev['davies_bouldin']:.3f}, ARI={ev['ari']:.3f}")
results.append({"method": "K-Means", "k": best_k, "silhouette": ev['silhouette'], "labels": kmeans_labels})

### 3. Gaussian Mixture Models

In [ ]:
best_gmm = None
best_sil = -1
best_k = 0

for k in range(2, 7):
    gmm = GaussianMixture(n_components=k, random_state=42)
    labels = gmm.fit_predict(X)
    ev = evaluate_clustering(labels, X, y_true)
    print(f"k={k}: Silhouette={ev['silhouette']:.3f}, DB={ev['davies_bouldin']:.3f}, ARI={ev['ari']:.3f}")
    if ev['silhouette'] > best_sil:
        best_sil = ev['silhouette']
        best_gmm = labels
        best_k = k

print(f"\nBest: k={best_k}, Silhouette={best_sil:.3f}")
results.append({"method": "GMM", "k": best_k, "silhouette": best_sil, "labels": best_gmm})


### 4. DBScan Clustering

In [ ]:
# OPSI 1: Normalisasi dulu biar skala 0-1
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

eps_values = [0.3, 0.5, 0.7, 1.0, 1.2, 1.5]
min_samples_values = [3, 5, 7, 10]

best_dbscan = None
best_sil = -1
best_params = None

for eps in eps_values:
    for min_pts in min_samples_values:
        dbscan = DBSCAN(eps=eps, min_samples=min_pts)
        labels = dbscan.fit_predict(X_scaled)  # pakai yang sudah dinormalisasi
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        if n_clusters >= 2:
            mask = labels != -1
            if sum(mask) > 0 and len(set(labels[mask])) >= 2:
                sil = silhouette_score(X_scaled[mask], labels[mask])
                db = davies_bouldin_score(X_scaled[mask], labels[mask])
                ari = adjusted_rand_score(y_true[mask], labels[mask]) if y_true is not None else -1
                print(f"eps={eps}, min_samples={min_pts}: clusters={n_clusters}, noise={n_noise}, sil={sil:.3f}, ARI={ari:.3f}")
                if sil > best_sil:
                    best_sil = sil
                    best_dbscan = labels
                    best_params = (eps, min_pts)

if best_dbscan is not None:
    print(f"\n✅ Best DBSCAN: eps={best_params[0]}, min_samples={best_params[1]}, Silhouette={best_sil:.3f}")
    results.append({"method": "DBSCAN", "k": len(set(best_dbscan)) - (1 if -1 in best_dbscan else 0), 
                    "silhouette": best_sil, "labels": best_dbscan})
else:
    print("❌ Tidak ada cluster valid dengan parameter tersebut")
    print("   Saran: Coba normalisasi data terlebih dahulu")

### Perbandingan Metode Clustering

In [ ]:
comparison = pd.DataFrame([{
    "Method": r["method"],
    "N_Clusters": r["k"],
    "Silhouette": r["silhouette"]
} for r in results])

print(comparison.sort_values("Silhouette", ascending=False))

best_method = comparison.loc[comparison["Silhouette"].idxmax()]
print(f"\n✅ METODE TERBAIK: {best_method['Method']} dengan {int(best_method['N_Clusters'])} cluster")
print(f"   Silhouette Score: {best_method['Silhouette']:.4f}")

# Ambil label dari metode terbaik
best_labels = None
for r in results:
    if r["method"] == best_method["Method"]:
        best_labels = r["labels"]
        break


### Exploratory Data Analysis (EDA) - Agglomerative Clustering

In [ ]:
df_clustered = df.copy()
df_clustered['Cluster'] = best_labels

# Scatter plot 2 fitur pertama
plt.figure(figsize=(10, 6))
scatter = plt.scatter(X[:, 0], X[:, 1], c=best_labels, cmap='viridis', s=50)
plt.colorbar(scatter, label='Cluster')
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title(f'Clustering - {best_method["Method"]}')
plt.show()


In [ ]:
# 1. Statistik per cluster
print(df_clustered.groupby('Cluster')[feature_names].mean())

# 2. Jumlah anggota
print(df_clustered['Cluster'].value_counts())

# 3. Scatter plot
plt.scatter(X[:, 0], X[:, 1], c=best_labels)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.show()

### Memberi Label pada Setiap Cluster

In [ ]:
main_feature = feature_names[0]
print(f"Berdasarkan fitur: {main_feature}")

# Filter noise untuk DBSCAN
if best_method["Method"] == "DBSCAN":
    df_clean = df_clustered[df_clustered['Cluster'] != -1].copy()
    print(f"(Noise diabaikan: {len(df_clustered) - len(df_clean)} data)")
else:
    df_clean = df_clustered.copy()

# Hitung rata-rata fitur utama per cluster
cluster_means = df_clean.groupby('Cluster')[main_feature].mean().sort_values()
sorted_clusters = cluster_means.index.tolist()
n_clusters = len(sorted_clusters)

# Assign label
label_mapping = {}
if n_clusters == 3:
    label_mapping = {sorted_clusters[0]: 'Rendah', sorted_clusters[1]: 'Sedang', sorted_clusters[2]: 'Tinggi'}
elif n_clusters == 2:
    label_mapping = {sorted_clusters[0]: 'Rendah', sorted_clusters[1]: 'Tinggi'}
else:
    third = n_clusters // 3
    for i, c in enumerate(sorted_clusters):
        if i < third: label_mapping[c] = 'Rendah'
        elif i < 2*third: label_mapping[c] = 'Sedang'
        else: label_mapping[c] = 'Tinggi'

df_clean['Kategori'] = df_clean['Cluster'].map(label_mapping)

print("\nPemetaan Cluster ke Kategori:")
for cluster, label in label_mapping.items():
    print(f"  Cluster {cluster} ({label}): rata-rata {main_feature} = {cluster_means[cluster]:.3f}")

print("\nHasil Akhir (10 data pertama):")
print(df_clean[feature_names[:4] + ['Cluster', 'Kategori']])

In [ ]:
# Visualisasi final
plt.figure(figsize=(10, 6))
color_map = {'Rendah': 'green', 'Sedang': 'orange', 'Tinggi': 'red'}
scatter_colors = df_clean['Kategori'].map(color_map)
plt.scatter(df_clean[feature_names[0]], df_clean[feature_names[1]], c=scatter_colors, s=50)
plt.xlabel(feature_names[0])
plt.ylabel(feature_names[1])
plt.title(f'Clustering {best_method["Method"]} - Kategori Wine')
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', label='Rendah'),
                   Patch(facecolor='orange', label='Sedang'),
                   Patch(facecolor='red', label='Tinggi')]
plt.legend(handles=legend_elements)
plt.show()

In [ ]:
print(f"""
- Metode Terbaik : {best_method['Method']}
- Jumlah Cluster : {int(best_method['N_Clusters'])}
- Silhouette Score: {best_method['Silhouette']:.4f}
- Label berdasarkan {main_feature}:
   - Tinggi  : Nilai tertinggi
   - Sedang  : Nilai menengah  
   - Rendah  : Nilai terendah
- Distribusi Kategori:""")
print(df_clean['Kategori'].value_counts())